# Example AI Risk ML Classifier with IBM Risk Atlas risks

A multi-label AI risk classification system that identifies applicable risks from AI use-case descriptions using the **IBM Risk Atlas** definitions via ai-atlas-nexus.

**Approach**: Hybrid scoring combining TF-IDF cosine similarity (vocabulary overlap) and domain keyword extraction (semantic coverage from ontology).

**Risk Taxonomy**: ~99 risks from the **IBM Risk Atlas**, or another preloaded taxonomy of your choice, or even your own set of risks loaded with the ai-atlas-nexus library.

---

## Setup

Install dependencies and import the classifier.

In [1]:
# Install required packages (uncomment if needed)
# !pip install scikit-learn numpy pandas jupyter ai-atlas-nexus

import sys
sys.path.insert(0, 'classifier')

from ai_risk_classifier import AIRiskClassifier
import pandas as pd
import numpy as np

## Initialize Classifier with IBM Risk Atlas

Create an instance of classifer. The risks are loaded dynamically from the toolkit.

In [2]:
# Initialize with IBM Risk Atlas from ai-atlas-nexus
clf = AIRiskClassifier.from_nexus(
    taxonomy='ibm-risk-atlas',     # Use IBM Risk Atlas (default)
    threshold_high=0.30,           # Score ≥ 0.30 → "high" relevance
    threshold_medium=0.12,         # Score ≥ 0.12 → "medium" relevance
    alpha=0.35,                    # Weight of TF-IDF (0–1); 1-alpha = weight of keywords
    aggregation="max"              # Max or mean across per-anchor TF-IDF scores
)

print(clf)
print(f"\nLoaded {len(clf.taxonomy)} risks from IBM Risk Atlas")

/Users/ingevejs/Documents/workspace/ingelise/risk-atlas-nexus-demos/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-08-12 10:43:50:800] - INFO - AIAtlasNexus - Created AIAtlasNexus instance. Base_dir: None


AIRiskClassifier(n_risks=99, vocab=5335, alpha=0.35, thresholds=(0.12, 0.3))

Loaded 99 risks from IBM Risk Atlas


In [3]:
# Build a taxonomy table from loaded risks
taxonomy_df = pd.DataFrame([
    {
        "Risk ID": r["id"],
        "Label": r["label"],
        "Category": r["group"],
        "Source": ", ".join(r["sources"][:1]) if r["sources"] else "N/A"
    }
    for r in clf.taxonomy
])

print(f"Total risks: {len(taxonomy_df)}\n")

   

Total risks: 99



## Example 1: Patient Q&A System

Analyze risks for an LLM answering patient questions about medications.

In [4]:
usecase_1 = "An LLM answers patient questions about medications and symptoms."

results_1 = clf.identify_risks_from_usecases([usecase_1])

print(results_1[0])


Use case: An LLM answers patient questions about medications and symptoms.
Identified 2 risk(s):
  [MEDIUM] Copyright infringement                     score=0.194  █████  kw=[questions ownership, new questions]
  [MEDIUM] Direct instructions attack                 score=0.158  ████  kw=[prompts questions, questions requests]


### Detailed Risk Breakdown for Patient Q&A

In [5]:
result = results_1[0]

# Separate by relevance level
print("\n=== HIGH RELEVANCE ===")
for risk in result.high:
    print(f"\n  {risk.label}")
    print(f"    Score: {risk.score:.4f} (TF-IDF: {risk.tfidf_score:.4f}, Keywords: {risk.keyword_score:.4f})")
    if risk.matched_keywords:
        print(f"    Matched keywords: {', '.join(risk.matched_keywords[:6])}")
    print(f"    Sources: {', '.join(risk.sources)}")

print("\n=== MEDIUM RELEVANCE ===")
for risk in result.medium:
    print(f"  {risk.label:<50} score={risk.score:.4f}")
    if risk.matched_keywords:
        print(f"    Keywords: {', '.join(risk.matched_keywords[:4])}")


=== HIGH RELEVANCE ===

=== MEDIUM RELEVANCE ===
  Copyright infringement                             score=0.1937
    Keywords: questions ownership, new questions
  Direct instructions attack                         score=0.1581
    Keywords: prompts questions, questions requests


## Example 2: Multiple Use Cases

Analyze several diverse AI applications in parallel.

In [6]:
usecases = [
    "An automated hiring system that screens job applications and ranks candidates.",
    "A generative AI assistant for creating marketing content and campaigns.",
    "A machine learning model for approving credit card applications.",
    "A medical AI system that diagnoses diseases from imaging.",
    "An LLM chatbot for customer support on an e-commerce platform.",
]

results = clf.identify_risks_from_usecases(usecases)

for i, result in enumerate(results, 1):
    print(result)


Use case: An automated hiring system that screens job applications and ranks candidates.
Identified 1 risk(s):
  [HIGH  ] Impact on Jobs                             score=0.308  █████████  kw=[job loss, job, automated, work automated]

Use case: A generative AI assistant for creating marketing content and campaigns.
Identified 3 risk(s):
  [MEDIUM] Spreading toxicity                         score=0.211  ██████  kw=[content, content toxic, toxic content, toxicity generative]
  [MEDIUM] Dangerous use                              score=0.137  ████  kw=[evaluated content, content properly]
  [MEDIUM] Spreading disinformation                   score=0.121  ███  kw=[disinformation generative]

Use case: A machine learning model for approving credit card applications.
Identified 2 risk(s):
  [MEDIUM] Impact on education: bypassing learning    score=0.265  ███████  kw=[learning, learning process, bypass learning, models]
  [MEDIUM] Unexplainable output                       score=0.235  █████

## Example 3: Risk Score Matrix

Get raw scores for all use cases and risks (useful for downstream analysis and sklearn pipelines).

In [7]:
# Get raw score matrix: (n_usecases, n_risks)
score_matrix = clf.score(usecases)

print(f"Score matrix shape: {score_matrix.shape}")
print(f"  Rows: {score_matrix.shape[0]} use cases")
print(f"  Cols: {score_matrix.shape[1]} risks\n")

# Create a detailed DataFrame
risk_ids = clf.risk_ids()
score_df = pd.DataFrame(
    score_matrix,
    columns=risk_ids,
    index=[f"UC{i+1}: {uc[:40]}..." for i, uc in enumerate(usecases)]
)

# Display rounded scores
display_df = score_df.copy()
display_df = display_df.round(4)
print(display_df)

Score matrix shape: (5, 99)
  Rows: 5 use cases
  Cols: 99 risks

                                                  evasion-attack  \
UC1: An automated hiring system that screens ...          0.0000   
UC2: A generative AI assistant for creating m...          0.0000   
UC3: A machine learning model for approving c...          0.0093   
UC4: A medical AI system that diagnoses disea...          0.0000   
UC5: An LLM chatbot for customer support on a...          0.0000   

                                                  impact-on-the-environment  \
UC1: An automated hiring system that screens ...                     0.0059   
UC2: A generative AI assistant for creating m...                     0.1030   
UC3: A machine learning model for approving c...                     0.0745   
UC4: A medical AI system that diagnoses disea...                     0.0086   
UC5: An LLM chatbot for customer support on a...                     0.0322   

                                                  

### Risk Distribution Analysis

Which risks are most common across the use cases?

In [8]:
# Average score per risk across all use cases
avg_scores = score_matrix.mean(axis=0)
risk_importance = pd.DataFrame({
    'Risk ID': risk_ids,
    'Avg Score': avg_scores,
    'Max Score': score_matrix.max(axis=0),
    'Min Score': score_matrix.min(axis=0),
})

risk_importance = risk_importance.sort_values('Avg Score', ascending=False)
print("\nTop risks by average score across all use cases:")
print(risk_importance.head(8).to_string(index=False))


Top risks by average score across all use cases:
                  Risk ID  Avg Score  Max Score  Min Score
           impact-on-jobs   0.084713   0.308467   0.005049
       bypassing-learning   0.066350   0.265245   0.005158
  reproducibility-agentic   0.054089   0.110232   0.000000
     unexplainable-output   0.053602   0.235514   0.000000
            dangerous-use   0.049251   0.136706   0.006695
       spreading-toxicity   0.045439   0.211007   0.000000
impact-on-the-environment   0.044833   0.102995   0.005926
  harmful-code-generation   0.041650   0.072007   0.000000


## Example 4: Custom Thresholds

Create a more sensitive classifier (lower thresholds = more risks detected).

In [9]:
# Strict classifier: lower thresholds catch more marginal risks
strict_clf = AIRiskClassifier.from_nexus(
    taxonomy='ibm-risk-atlas',   
    threshold_high=0.20,
    threshold_medium=0.08,
    alpha=0.4,
)

strict_results = strict_clf.identify_risks_from_usecases(
    ["A chatbot for general customer service."]
)

print("\n=== STRICT CLASSIFIER ===")
print(strict_results[0])

print(f"\nTotal risks detected (strict): {len(strict_results[0].risks)}")
print(f"  High: {len(strict_results[0].high)}, Medium: {len(strict_results[0].medium)}")

[2026-08-12 10:43:51:768] - INFO - AIAtlasNexus - Created AIAtlasNexus instance. Base_dir: None



=== STRICT CLASSIFIER ===

Use case: A chatbot for general customer service.
Identified 2 risk(s):
  [MEDIUM] Model usage rights restrictions            score=0.120  ███  kw=[service licenses]
  [MEDIUM] Data usage rights restrictions             score=0.118  ███  kw=[service license]

Total risks detected (strict): 2
  High: 0, Medium: 2


## Example 5: Keyword Matching Deep Dive

Understand how keyword matching contributes to risk scores.

In [10]:
usecase = "A system that automatically decides parole recommendations for incarcerated individuals."
results_parole = clf.identify_risks_from_usecases([usecase])

print(results_parole[0])

print("\n=== KEYWORD ANALYSIS ===")
for risk in results_parole[0].high:
    print(f"\n{risk.label}")
    print(f"  Score: {risk.score:.4f}")
    print(f"  TF-IDF: {risk.tfidf_score:.4f} | Keywords: {risk.keyword_score:.4f}")
    print(f"  Matched keywords: {risk.matched_keywords}")


Use case: A system that automatically decides parole recommendations for incarcerated individuals.
Identified 4 risk(s):
  [MEDIUM] Incomplete advice                          score=0.163  ████  kw=[recommendations, advice recommendations]
  [MEDIUM] Output bias                                score=0.163  ████  kw=[individuals bias, groups individuals]
  [MEDIUM] Attribute inference attack                 score=0.147  ████  kw=[inferred individuals]
  [MEDIUM] Data privacy rights alignment              score=0.145  ████  kw=[individuals seemingly, reidentification individuals]

=== KEYWORD ANALYSIS ===


## Example 6: Exporting Results

Convert results to JSON/dict for integration with other systems.

In [11]:
import json

# Convert to structured format
result_dict = results_parole[0].to_dict()

print(json.dumps(result_dict, indent=2)[:500] + "...")

# Export all results as DataFrame
all_findings = []
for result in results:
    for risk in result.risks:
        all_findings.append({
            'usecase': result.usecase[:50],
            'risk_id': risk.id,
            'risk_label': risk.label,
            'category': risk.group,
            'relevance': risk.relevance,
            'score': risk.score,
            'tfidf_score': risk.tfidf_score,
            'keyword_score': risk.keyword_score,
        })

findings_df = pd.DataFrame(all_findings)
print(f"\nExported {len(findings_df)} risk findings across {len(usecases)} use cases")
print(findings_df[findings_df['relevance'] == 'high'].head(10))

{
  "usecase": "A system that automatically decides parole recommendations for incarcerated individuals.",
  "risks": [
    {
      "id": "incomplete-advice",
      "label": "Incomplete advice",
      "group": "output",
      "relevance": "medium",
      "score": 0.1628,
      "matched_keywords": [
        "recommendations",
        "advice recommendations"
      ],
      "sources": [
        "ibm-risk-atlas"
      ]
    },
    {
      "id": "output-bias",
      "label": "Output bias",
      "gr...

Exported 6 risk findings across 5 use cases
                                             usecase         risk_id  \
0  An automated hiring system that screens job ap...  impact-on-jobs   

       risk_label       category relevance   score  tfidf_score  keyword_score  
0  Impact on Jobs  non-technical      high  0.3085       0.1385            0.4  


## Summary

The AI Risk Classifier provides a way to build an AI risk identification ml classifier using the **IBM Risk Atlas** outputs:

### Key Features
- **Live ontology**: Works with ~99 risks from the IBM Risk Atlas via ai-atlas-nexus
- **No training required**: Works out-of-the-box with risk descriptions and extracted keywords
- **Interpretable**: Hybrid scoring combines TF-IDF similarity + domain keyword extraction
- **Configurable**: Adjust thresholds and weighting to match your risk tolerance
- **Extensible**: Works with any risk taxonomy from ai-atlas-nexus (NIST, MIT, OWASP, etc.)
- **sklearn-compatible**: `.score()` method returns matrices suitable for ML pipelines

### Possible Use Cases
1. **AI audit workflows**: Quickly identify risks in proposed AI applications
2. **Risk inventory**: Screen multiple use cases for comparative risk assessment
3. **Compliance checks**: Automatic flagging of high-risk domains (healthcare, finance, hiring)
4. **Threshold calibration**: Benchmark against labelled data to optimize for your deployment
5. **Downstream ML**: Integrate scores as features in broader governance systems

### Supported Taxonomies
You can initialize the classifier with different taxonomies from ai-atlas-nexus:
```python
AIRiskClassifier.from_nexus(taxonomy='nist-ai-rmf')      # NIST
AIRiskClassifier.from_nexus(taxonomy='mit-ai-risk-repository')  # MIT
AIRiskClassifier.from_nexus(taxonomy='owasp-llm-2.0')    # OWASP
AIRiskClassifier.from_nexus(taxonomy='ibm-risk-atlas')   # IBM (default)
```
